# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
pop_products = cat[cat["product_name"].str.lower().str.contains("pop")].copy()
print("count:",len(pop_products))
pop_products

count: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
print(pop_products["product_name"].str.lower().str.contains("strawberry").sum())

4


**Pop-Tart rows that are a different product:**
Every listed Pop-Tart is strawberry flavored, so they are all the same product.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
mapping = {
    "PT-12":"PT_STRAWBERRY_4",
    "SKU#459812":"PT_STRAWBERRY_4",
    "POPTART-STRAW":"PT_STRAWBERRY_6",
    "PT_STRAWBERRY":"PT_STRAWBERRY_4",
}
pop_products["sku"] = pop_products["sku"].map(mapping).fillna(cat["sku"])

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print("unique values before:", cat["category"].nunique())
cat["category"]=cat["category"].str.strip().str.title()
print("unique values after:", cat["category"].nunique())
print(cat["category"].unique())
print("\nPairs that are 2 labels for the same thing are:\n1. breakfast and breakfast food \n2. emergency supplies and emergency\n3. alcoholic beverages and alcohol\n4  baby and baby products")

unique values before: 27
unique values after: 15
['Breakfast Foods' 'Breakfast' 'Emergency Supplies' 'Emergency'
 'Alcoholic Beverages' 'Alcohol' 'Entertainment' 'Electronics' 'Groceries'
 'Grocery' 'Health' 'Snacks' 'Household' 'Baby' 'Baby Products']

Pairs that are 2 labels for the same thing are:
1. breakfast and breakfast food 
2. emergency supplies and emergency
3. alcoholic beverages and alcohol
4  baby and baby products


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
cat["product_name"] = cat["product_name"].astype("string").str.replace(r"[^\w\s]", "", regex=True).str.lower().str.split()
cat["product_name"] = cat["product_name"].apply(sorted).str.join(" ")
more_skus = cat.groupby("product_name").agg(n_skus=("sku","nunique"),skus=("sku", "unique")).query("n_skus > 1")
print(len(more_skus))

more_skus

13


,n_skus,skus
product_name,,
24pack beer light,2,"[BEER-24, LTBEER24]"
4pack aa batteries,2,"[BAT-AA4, BATT_AA_4PK]"
board game monopoly,3,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bread white,2,"[BREAD-WH, WHITEBREAD]"
butter peanut,2,"[PB-16, PNUTBTR-16]"
candles emergency,2,"[CNDL-EM, CANDLE-EMRG]"
canned soup,2,"[SOUP-CAN, CNSOUP-01]"
cards playing,2,"[CARDS-01, PLAYCARD-STD]"
chips pack variety,2,"[CHIPS-VP, CHIPVARIETY]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
rows_before = len(cat)
cat["sku"]=cat["sku"].map(mapping).fillna(cat["sku"])
assert cat["sku"].notna().all(), "produced nulls"
assert len(cat) == rows_before, "number of rows not the same"
print("assertions passed")
cat

assertions passed


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT_STRAWBERRY_4,poptarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,PT_STRAWBERRY_4,poptart strawberry,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,PT_STRAWBERRY_6,poptarts strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY_4,poptarts strawberry,Breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,24pk bottled water,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack
5,WATER24PK,24pack bottled water,Emergency,Bottled,2.38,Sara Lee,24 pack
6,WTR_24PK,24 pack water,Emergency,NaN,2.08,Frito-Lay,1 pack
7,BATT-100,4pk aa batteries,Emergency Supplies,AA,NaN,3M Company,6 pack
8,BAT-AA4,4pack aa batteries,Emergency Supplies,AA,4.16,Kimberly-Clark,4pk
9,BATT_AA_4PK,4pack aa batteries,Emergency,AA,3.71,Anheuser-Busch,24 pack


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a. The packs of beers also appears fragmented, as they have different product names and skus. I would differentiate the number of bottles in each pack then assign them to the same SKU,category, and product number.  Similarly, the peanut butter is also fragmented.  I would also assign them to the same SKU, category, and product number.

b. Consolidation stops being a data-cleaning task when products are not separated by case sensitivity, spacing, etc., but by the words in the labels themselves.  For example, I am geniunely not sure if infant formula and baby formula are the same product.  If I was working to consolidate this data, I would ask the client or the store manager.